# Agent 365 - Registry Ingester (Fabric) — recommended default

> **Status: supported (app-only, unattended) — this is the recommended default path** for landing the
> Agent 365 registry into the Lakehouse. As of **PAX `purview-v1.11.11`** and the current
> Microsoft Graph docs, the *Agent 365 catalog* endpoint supports **application permissions** and
> **`/v1.0`**, so this notebook runs **headless on a schedule** with a service principal — no
> interactive sign-in. (It was previously a delegated-only, interactive PREVIEW.)

## What this does

Pulls the tenant's **Agent 365 agent catalogue** (declarative agents, plugins, etc. - including
their data-access **capabilities/permissions**: OneDrive/SharePoint read, Graph connector, code
interpreter, image generation, uploaded files) into the Lakehouse Delta table `dbo.agents_365`
(the table the dashboard reads), keyed on `Title ID` (the `T_`-prefixed titleId).

## Requirements & caveats (read before using)

| Item | Detail |
|---|---|
| **Auth** | **App-only / client credentials** (service principal). Runs unattended. No user sign-in. |
| **Permissions** | **Application** permissions `CopilotPackages.Read.All` **+** `Application.Read.All`, **admin-consented**. |
| **Agent 365 licence** | **Still required in the tenant.** This is a *SKU* check, separate from permissions - a missing licence returns **`403`** (`Customer must be licensed for Agent 365`). |
| **Endpoint version** | Uses **`/v1.0`** (GA). Automatically falls back to **`/beta`** if a tenant hasn't surfaced v1.0 yet (PAX itself still calls beta). |
| **Point-in-time only** | No history; deleted agents disappear; `Date created`/`Created by` need a separate Purview audit-log join (out of scope here). |

**Relationship to the export lander:** `Copilot_Agent365_Lander.ipynb` (admin-center **export CSV**)
is a **fallback** for tenants that can't grant the app-registration permissions this notebook needs,
or for one-off / evaluation runs. **Prefer this notebook** for scheduled production pipelines — you
get the live capability detail, no CSV upload step, and an automated refresh. The two notebooks are
**alternatives** — they write to the same `dbo.agents_365` table, so running both in the same pipeline
would just clobber each other.

## Setup

- An **app registration** (service principal) with the two **Application** permissions above,
  admin-consented, plus a **client secret** (store it in Key Vault / a Fabric secret, not in code)
  or a **certificate** / **managed identity**.
- Install MSAL only if you use the managed-identity path; the default client-secret path uses
  `requests` and needs nothing extra.

*Endpoint, 28-column schema and capability fields align with the PAX Agent 365 enrichment output so
the dashboard's `Agents 365` query reads them unchanged. Ref: Microsoft Graph
`copilot/admin/catalog/packages` (v1.0) and PAX `purview-v1.11.11`.*

## 1. Configuration & app-only sign-in

**App-only (client-credentials)** flow - runs unattended, so this notebook can be a scheduled Fabric
job. No device-code, no browser. The service principal's admin-consented **Application** permissions
(`CopilotPackages.Read.All` + `Application.Read.All`) are carried in the token via the `.default`
scope.

In [ ]:
# === CONFIG ===
TENANT_ID    = '<your-tenant-guid>'
CLIENT_ID    = '<app-reg-client-id>'      # Application perms: CopilotPackages.Read.All + Application.Read.All (admin-consented)
TARGET_TABLE = 'dbo.agents_365'   # canonical table the dashboard reads (matches the lander)
WRITE_MODE   = 'overwrite'

# Client secret - DO NOT hardcode. Pull from Key Vault / a Fabric-managed secret at runtime, e.g.:
#   CLIENT_SECRET = notebookutils.credentials.getSecret('https://<your-vault>.vault.azure.net/', 'Agent365AppSecret')
CLIENT_SECRET = '<from-key-vault>'

# App-only token via client credentials (mirrors Copilot_Audit_Log_Direct_Ingester.ipynb).
import requests

def _get_graph_token() -> str:
    url  = f'https://login.microsoftonline.com/{TENANT_ID}/oauth2/v2.0/token'
    data = {
        'client_id':     CLIENT_ID,
        'client_secret': CLIENT_SECRET,
        'scope':         'https://graph.microsoft.com/.default',
        'grant_type':    'client_credentials',
    }
    r = requests.post(url, data=data)
    r.raise_for_status()
    return r.json()['access_token']

# Managed-identity alternative (no secret) - uncomment if the notebook runs under a UAMI/SAMI:
#   from azure.identity import DefaultAzureCredential
#   TOKEN = DefaultAzureCredential().get_token('https://graph.microsoft.com/.default').token

TOKEN = _get_graph_token()
print('app-only token acquired:', bool(TOKEN))

## 2. Call the v1.0 catalog endpoint

Uses **`/v1.0`** (GA, app-only) and **auto-falls back to `/beta`** if v1.0 isn't yet exposed in the
tenant. List -> page via `@odata.nextLink` -> fetch per-package detail (the `elementDetails`
capability fields only appear at the detail level).

- **`403`** = missing **Agent 365 licence** in the tenant (SKU check), *or* the app hasn't been
  admin-consented for `CopilotPackages.Read.All`.
- **`401`** = token/consent problem, not a code bug.

In [ ]:
import requests

API_VERSION = 'v1.0'   # GA + app-only; falls back to beta below if a tenant hasn't surfaced v1.0 yet
def _base(v):
    return f'https://graph.microsoft.com/{v}/copilot/admin/catalog/packages'
BASE = _base(API_VERSION)
H = {'Authorization': f'Bearer {TOKEN}'}

# Probe, with automatic v1.0 -> beta fallback (PAX itself still calls beta)
probe = requests.get(f'{BASE}?$top=1', headers=H)
if probe.status_code == 404 and API_VERSION == 'v1.0':
    API_VERSION = 'beta'
    BASE = _base(API_VERSION)
    probe = requests.get(f'{BASE}?$top=1', headers=H)
if probe.status_code == 403:
    raise PermissionError('403: Agent 365 catalog not accessible. Requires an Agent 365 LICENCE in '
                          'the tenant (SKU check) AND admin-consented Application permission '
                          'CopilotPackages.Read.All.')
if probe.status_code == 401:
    raise PermissionError('401: token/consent problem - check the app has admin consent for '
                          'CopilotPackages.Read.All + Application.Read.All (Application permissions).')
probe.raise_for_status()
print(f'Using {API_VERSION} endpoint.')

# Page the catalog list
packages, url = [], BASE
for _ in range(500):                       # safety cap on paging
    r = requests.get(url, headers=H); r.raise_for_status()
    data = r.json(); packages.extend(data.get('value', []))
    url = data.get('@odata.nextLink')
    if not url:
        break
print('packages in catalog:', len(packages))

## 3. Map to the 28-column schema → Delta

Column names match the PAX `ConvertTo-Agent365Row` output so the dashboard's `Agents 365` query can
read them. `Title ID` is the primary/merge key. `Date created` / `Created by` are left blank here
(they need a separate Purview audit-log join — out of scope for this preview).

In [ ]:
# === SHAPE CATALOG -> canonical agents_365 schema ==============================
# Field names below are verified against a live Agent 365 catalog response, not
# assumed. The list endpoint returns 24 fields; the per-package detail endpoint
# adds usage metrics and elementDetails. Anything the API genuinely does not
# provide is written as an empty string rather than guessed, so the dashboard
# shows blank instead of wrong.
import json as _json
from pyspark.sql import functions as F


def _join(v):
    """Flatten a list/scalar to a semicolon string."""
    if isinstance(v, list):
        return ';'.join(
            _json.dumps(x, ensure_ascii=False) if isinstance(x, (dict, list)) else str(x)
            for x in v if x is not None
        )
    return '' if v is None else str(v)


def _access(v):
    """packageAccessEntity collections -> readable names."""
    if not isinstance(v, list):
        return ''
    out = []
    for e in v:
        if isinstance(e, dict):
            out.append(e.get('displayName') or e.get('id') or _json.dumps(e))
        else:
            out.append(str(e))
    return ';'.join(out)


def _elements(detail):
    """
    elementDetails is [{elementType, elements:[{id, definition}]}] where
    `definition` is a JSON string. Return (types, bot_ids, command_titles).
    The previous version treated this as a flat dict, which silently produced
    empty values for every capability column.
    """
    types, bots, cmds = [], [], []
    for grp in detail.get('elementDetails') or []:
        if not isinstance(grp, dict):
            continue
        et = grp.get('elementType')
        if et:
            types.append(str(et))
        for el in grp.get('elements') or []:
            if not isinstance(el, dict):
                continue
            raw = el.get('definition')
            if not raw:
                continue
            try:
                d = _json.loads(raw) if isinstance(raw, str) else raw
            except Exception:
                continue
            if isinstance(d, dict):
                if d.get('botId'):
                    bots.append(str(d['botId']))
                for cl in d.get('commandLists') or []:
                    for c in (cl or {}).get('commands') or []:
                        if isinstance(c, dict) and c.get('title'):
                            cmds.append(str(c['title']))
                for c in d.get('commands') or []:
                    if isinstance(c, dict) and c.get('title'):
                        cmds.append(str(c['title']))
    return (';'.join(dict.fromkeys(types)),
            ';'.join(dict.fromkeys(bots)),
            ';'.join(dict.fromkeys(cmds)))


rows, detail_failures = [], 0
for pkg in packages:
    pid = pkg.get('id') or pkg.get('titleId') or pkg.get('packageId')
    detail = pkg
    try:
        d = requests.get(f'{BASE}/{pid}', headers=H, timeout=60)
        if d.ok:
            detail = d.json()
        else:
            detail_failures += 1
    except Exception:
        detail_failures += 1

    el_types, bot_ids, commands = _elements(detail)
    tid = detail.get('id') or detail.get('titleId') or detail.get('packageId') or ''
    title_id = tid if str(tid).startswith(('T_', 'P_')) else f'T_{tid}'

    rows.append({
        # --- identity -----------------------------------------------------
        'Agent name':        detail.get('displayName') or '',
        'Title ID':          title_id,
        'Version':           detail.get('version') or '',
        # agentIdentityId IS the Entra Agent ID. Populated for Entra-registered
        # (Copilot Studio) agents only; blank for store/1P packages.
        'Entra Agent ID':    detail.get('agentIdentityId') or '',
        'Bot Id':            bot_ids,
        'App Id':            detail.get('appId') or '',
        'Asset Id':          detail.get('assetId') or '',

        # --- provenance ---------------------------------------------------
        # 'publisher' is the publishing company; 'ownerId' is the tenant user who
        # created it. Both are exposed so neither is inferred from the other.
        'Publisher':         detail.get('publisher') or '',
        'Agent creator':     detail.get('publisher') or '',
        'Agent creator ID':  detail.get('ownerId') or '',
        'Agent type (A365)': detail.get('type') or '',
        'Created in':        detail.get('platform') or '',
        'Date created':      detail.get('createdDateTime') or '',
        'Last updated':      detail.get('lastModifiedDateTime') or '',

        # --- description --------------------------------------------------
        'Agent description': detail.get('shortDescription') or detail.get('longDescription') or '',
        'Categories':        _join(detail.get('categories')),

        # --- distribution / status ----------------------------------------
        'Supported in':      _join(detail.get('supportedHosts')),
        'Availability':      detail.get('availableTo') or '',
        'Status':            detail.get('deployedTo') or '',
        'Is Blocked':        str(detail.get('isBlocked', '')),
        'Users shared':      _access(detail.get('sharedWithUsersAndGroups')),
        'Groups shared':     _access(detail.get('allowedUsersAndGroups')),

        # --- capability ----------------------------------------------------
        'Element types':     el_types or _join(detail.get('elementTypes')),
        'Custom actions':    commands,

        # --- usage (detail endpoint only) ----------------------------------
        'Active Users':      str(detail.get('activeUsers', '') or ''),
        'Total sessions':    str(detail.get('totalSessions', '') or ''),
        'Exception rate':    str(detail.get('exceptionRate', '') or ''),
        'Run Time':          str(detail.get('totalRunTimeInHours', '') or ''),
        'Last Activity Date': detail.get('lastUsedDateTime') or '',

        # --- not exposed by this API --------------------------------------
        # These come from the Admin Center CSV export, not the catalog API.
        # Written blank rather than guessed so the dashboard degrades honestly.
        'Sensitivity': '',
        'Can read OneDrive and Sharepoint items': '', 'OneDrive and Sharepoint items': '',
        'Can read OneDrive files': '', 'OneDrive files': '', 'OneDrive sites': '',
        'Can read Sharepoint sites and files': '', 'Sharepoint files': '', 'Sharepoint sites': '',
        'Can extend to Graph connector': '', 'Graph connector details': '',
        'Can generate images using user prompt': '', 'Can use code interpreter': '',
        'Contains uploaded files': '', 'Uploaded files': '',
        'Environment Id': '', 'Instructions': '', 'Deployment': '', 'Risks': '',
    })

# Everything as string for a schema-stable Delta write.
rows = [{k: ('' if v is None else str(v)) for k, v in r.items()} for r in rows]

# The lander writes the same canonical column set. Keeping the two producers
# aligned means whichever runs last leaves the table the same shape.
CANONICAL = [
    'Agent name', 'Supported in', 'Date created', 'Agent creator', 'Publisher',
    'Agent type (A365)', 'Version', 'Availability', 'Agent creator ID',
    'Agent description', 'Created in', 'Last updated', 'Custom actions',
    'Title ID', 'Sensitivity',
    'Can read OneDrive and Sharepoint items', 'OneDrive and Sharepoint items',
    'Can read OneDrive files', 'OneDrive files', 'OneDrive sites',
    'Can read Sharepoint sites and files', 'Sharepoint files', 'Sharepoint sites',
    'Can extend to Graph connector', 'Graph connector details',
    'Can generate images using user prompt', 'Can use code interpreter',
    'Contains uploaded files', 'Uploaded files', 'Status',
    'Active Users', 'Total sessions', 'Exception rate', 'Last Activity Date',
    'Deployment', 'Run Time', 'Risks',
]
for r in rows:
    for c in CANONICAL:
        r.setdefault(c, '')

if rows:
    ordered = CANONICAL + [c for c in rows[0] if c not in CANONICAL]
    df = spark.createDataFrame(rows).select(*[F.col(f'`{c}`') for c in ordered])
else:
    df = spark.createDataFrame([], ','.join(f'`{c}` string' for c in CANONICAL))

# ---- ingestion report ---------------------------------------------------
filled = {c: sum(1 for r in rows if r.get(c)) for c in (rows[0] if rows else {})}
print(f'packages shaped : {len(rows)}')
if detail_failures:
    print(f'detail fetch failed for {detail_failures} package(s) - list fields used instead')
print('populated columns:')
for c in CANONICAL:
    n = filled.get(c, 0)
    if n:
        print(f'   {c:42} {n}/{len(rows)}')
blank = [c for c in CANONICAL if not filled.get(c)]
if blank:
    print(f'blank ({len(blank)}): {", ".join(blank)}')
    print('   -> these are not exposed by the catalog API; use the Admin Center')
    print('      CSV export via Copilot_Agent365_Lander to populate them.')

(df.write.mode(WRITE_MODE)
   .option('overwriteSchema', 'true')
   .option('delta.columnMapping.mode', 'name')
   .option('delta.minReaderVersion', '2')
   .option('delta.minWriterVersion', '5')
   .format('delta').saveAsTable(TARGET_TABLE))
print(f'wrote -> {TARGET_TABLE} ({WRITE_MODE}) | columns: {len(df.columns)}')
